In [ ]:
#| default_exp env

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *
from tempfile import TemporaryDirectory

The values a project needs to build and ship, and the environment its commands run in.

`strip_bundle` and `venv_env` build the environment a child process is given. `EnvStore` holds the
keys a step names in `needs`. dockeasy stores those, and this module imports it only when a value is
read or written, so a pipeline that runs `pytest` and nothing else never pays for it.

In [ ]:
#| export
from __future__ import annotations
import asyncio, inspect, os, sys, threading
from fastcore.all import Path

In [ ]:
#| export
BUNDLE_ONLY = ('PYTHONHOME', 'PYTHONPATH', 'PYTHONEXECUTABLE', '__PYVENV_LAUNCHER__', 'RESOURCEPATH')

class EnvError(RuntimeError): pass

`BUNDLE_ONLY` names the variables py2app and py2exe set to point an interpreter at the bundle's own
copy of Python. A frozen app inherits them from its launcher, and anything it spawns has to lose
them.

`EnvError` is what `EnvStore` raises: a store with no dockeasy behind it, a key with no name, a
value with no content.

In [ ]:
#| export
def strip_bundle(env, frozen=None):
    """`env` without a frozen host's interpreter redirection, unchanged where there is none.

    py2app and py2exe point `PYTHONHOME` and `PYTHONPATH` at the bundle so its own helper starts.
    A child that keeps them imports the bundle's standard library under another interpreter, which
    fails somewhere unrelated: `nbdev-test` run from a bundled app died inside `linecache` before
    reporting a single notebook. Outside a bundle this returns what it was given, untouched.
    """
    if not (getattr(sys, 'frozen', False) if frozen is None else frozen): return env
    for name in BUNDLE_ONLY: env.pop(name, None)
    env['PYTHONUTF8'] = '1'
    return env

def clean_env():
    "This process's environment, safe to hand to a child."
    return strip_bundle(os.environ.copy(), frozen=True)

`strip_bundle` edits the mapping it is given and returns that same mapping. It makes no copy, which
is why `clean_env` hands it `os.environ.copy()` rather than `os.environ`. Outside a bundle the
mapping comes back untouched, so a caller that claimed nothing about an environment still claims
nothing.

`frozen` overrides the `sys.frozen` check. That is what makes the bundle branch reachable from a
notebook that is not itself frozen. `clean_env` passes `frozen=True` rather than consulting
`sys.frozen`, because a caller asking for an environment safe to hand to a child wants one either
way.

`PYTHONUTF8` is set on the way out. An interpreter that lost `PYTHONHOME` also lost the encoding its
launcher chose for it.

In [ ]:
env = {'PYTHONHOME': '/Pullup.app/Contents/Resources', 'PATH': '/usr/bin', 'HOME': '/Users/me'}
strip_bundle(dict(env), frozen=True)

{'PATH': '/usr/bin', 'HOME': '/Users/me', 'PYTHONUTF8': '1'}

In [ ]:
#| hide
test_eq(strip_bundle(dict(env), frozen=False), env)
out = strip_bundle(dict(env), frozen=True)
assert not (set(BUNDLE_ONLY) & set(out)), 'every redirection name is gone'
test_eq(out['HOME'], '/Users/me')
same = dict(env)
test_is(strip_bundle(same, frozen=True), same)

In [ ]:
#| export
def venv_env(python=None, env=None):
    """`env` (this process's, by default) with `python`'s virtual environment in front of it.

    `python` is a path to an interpreter, which is how a caller says "run this project's commands
    in this project's environment". Passing None claims nothing about the environment beyond the
    bundle hygiene above.
    """
    env = strip_bundle(dict(os.environ if env is None else env))
    if not python: return env
    bindir = str(Path(python).parent)
    env['VIRTUAL_ENV'] = str(Path(bindir).parent)
    # Not `UV_PROJECT_ENVIRONMENT`. It is read wherever the process ends up rather than where it
    # started, so a `uv sync` run in another checkout syncs that project's lock into this venv and
    # prunes everything the lock does not name. uv finds the right environment from the directory.
    env.pop('UV_PROJECT_ENVIRONMENT', None)
    env['PATH'] = bindir + os.pathsep + env.get('PATH', '')
    env.pop('PYTHONHOME', None)
    return env

`venv_env` puts an interpreter's virtual environment in front of an environment. It sets
`VIRTUAL_ENV` to the directory above the interpreter's, prepends the interpreter's directory to
`PATH`, and drops `PYTHONHOME`. The mapping it is given is copied, so the caller's own dict is never
edited.

`PATH` is prepended, not replaced. What was already there stays reachable behind the venv.

`UV_PROJECT_ENVIRONMENT` is removed rather than set, and the comment in the source says why.

In [ ]:
e = venv_env('/repo/.venv/bin/python', env={'PATH': '/usr/bin', 'PYTHONHOME': '/frozen'})
e['VIRTUAL_ENV'], e['PATH'], 'PYTHONHOME' in e

('/repo/.venv', '/repo/.venv/bin:/usr/bin', False)

In [ ]:
#| hide
given = {'PATH': '/usr/bin', 'UV_PROJECT_ENVIRONMENT': '/elsewhere'}
out = venv_env('/repo/.venv/bin/python', env=given)
test_eq(given, {'PATH': '/usr/bin', 'UV_PROJECT_ENVIRONMENT': '/elsewhere'})
assert 'UV_PROJECT_ENVIRONMENT' not in out
test_eq(out['PATH'].split(os.pathsep), ['/repo/.venv/bin', '/usr/bin'])
test_eq(venv_env(None, env={'PATH': '/usr/bin'}), {'PATH': '/usr/bin'})

In [ ]:
#| export
def _dockeasy():
    try: from dockeasy.core import env_get, env_set, secret_get, secret_set
    except ImportError as e:
        raise EnvError('environment values are stored by dockeasy: pip install "pullup[cloud]"') from e
    import logging
    logging.getLogger('dotenv.main').setLevel(logging.ERROR)
    return env_get, env_set, secret_get, secret_set

def _call(fn, *args, **kwargs):
    "Call `fn`, awaiting it on its own loop when dockeasy hands back a coroutine."
    value = fn(*args, **kwargs)
    if not inspect.isawaitable(value): return value
    try: asyncio.get_running_loop()
    except RuntimeError: return asyncio.run(value)
    out = []
    def run():
        try: out.append((True, asyncio.run(value)))
        except BaseException as e: out.append((False, e))
    thread = threading.Thread(target=run); thread.start(); thread.join()
    ok, value = out[0]
    if ok: return value
    raise value

`_dockeasy` imports the store's four functions on first use, and raises `EnvError` naming the extra
that installs them. It also quiets the `dotenv.main` logger, which warns for every key it is asked
for and does not find. That is the ordinary case for a project that has not set one yet.

`_call` runs one of those four whatever shape it has. dockeasy's are synchronous today and the value
comes straight back. A coroutine is run to completion on a new loop. Where a loop is already running
in this thread, that new loop is run in a new thread, because `asyncio.run` refuses to nest. An
exception raised inside that thread is re-raised in the caller, not swallowed.

A notebook cell already runs under a loop, so the example below takes the thread. Both branches give
the same answer.

In [ ]:
async def double(x): return x*2
_call(double, 21), _call(lambda x: x*2, 21)

(42, 42)

In [ ]:
#| hide
async def refused(): raise EnvError('the store said no')
test_fail(lambda: _call(refused), contains='the store said no')
out = []
def no_loop_here(): out.append(_call(double, 21))
t = threading.Thread(target=no_loop_here); t.start(); t.join()
test_eq(out, [42])

In [ ]:
#| export
class EnvStore:
    "The environment keys one project cares about, read and written through dockeasy."
    def __init__(self, service='fastops', path=None): self.service, self.path = service, path
    def get(self, key, secret=True):
        "One value, from the keychain or the env file, falling back to this process's environment."
        env_get, _set, secret_get, _sset = _dockeasy()
        stored = (_call(secret_get, key, service=self.service, path=self.path) if secret
                  else _call(env_get, key, path=self.path))
        return stored or os.environ.get(key) or ''
    def set(self, key, value, secret=True):
        "Store one value. A secret goes to the keychain as well as the file; a variable does not."
        key = str(key or '').strip()
        if not key: raise EnvError('a key is required')
        if not str(value): raise EnvError(f'{key} needs a value')
        env_get, env_set, secret_get, secret_set = _dockeasy()
        if secret: _call(secret_set, key, str(value), service=self.service, path=self.path)
        else: _call(env_set, key, str(value), path=self.path)
        return {'key': key, 'secret': bool(secret)}
    def unset(self, key):
        "Remove a value from every place `set` put it. Says which places actually held one."
        from dockeasy.core import _FASTOPS_ENV
        from dotenv import unset_key
        key, gone = str(key), []
        try:
            import keyring
            if keyring.get_password(self.service, key) is not None:
                keyring.delete_password(self.service, key)
                gone.append('keychain')
        except Exception: pass
        path = str(self.path or _FASTOPS_ENV)
        try:
            removed, _ = unset_key(path, key)
            if removed: gone.append('env file')
        except Exception: pass
        os.environ.pop(key, None)
        return {'key': key, 'removed': gone}
    def values(self, keys, secret=True):
        "Every key that has a value, as a dict. A key with none is absent rather than empty."
        out = {}
        for k in keys:
            try: v = self.get(k, secret=secret)
            except Exception: v = os.environ.get(k) or ''
            if v: out[k] = v
        return out

`EnvStore` is the set of environment keys one project cares about. `service` names the keychain
service dockeasy stores secrets under. `path` names the env file; the default is dockeasy's own,
under `~/.config/fastops`.

`get` reads the keychain when `secret` is true and the env file when it is false, then falls back to
this process's environment. A key with a value nowhere reads as `''`, never None.

`set` writes a secret to the keychain and to the env file, and a variable to the env file alone. A
blank key raises `EnvError`, and so does an empty value. Both would otherwise store an entry no step
can use and fail much later.

`unset` removes a key from every place `set` put it, and reports the places that held one. It drops
the key from this process's environment as well, which `removed` never names.

`values` returns only the keys that have a value. A key with none is absent rather than present and
empty, so a step can ask what it is missing by set difference. A key whose read raises is not an
error either; the process environment answers for it, and it is absent when that is empty too.

In [ ]:
#| hide
tmp = TemporaryDirectory()
envfile = Path(tmp.name)/'.env'

In [ ]:
store = EnvStore(path=envfile)
store.set('PULLUP_DEMO_REGISTRY', 'ghcr.io/example', secret=False)
store.get('PULLUP_DEMO_REGISTRY', secret=False)

'ghcr.io/example'

Nothing above is a secret, and nothing below prints one. A secret takes the same two calls with
`secret=True`, and reaches the keychain as well as the file.

In [ ]:
store.values(['PULLUP_DEMO_REGISTRY', 'PULLUP_DEMO_TAG'], secret=False)

{'PULLUP_DEMO_REGISTRY': 'ghcr.io/example'}

In [ ]:
store.unset('PULLUP_DEMO_REGISTRY')

{'key': 'PULLUP_DEMO_REGISTRY', 'removed': ['env file']}

In [ ]:
#| hide
test_eq(store.get('PULLUP_DEMO_REGISTRY', secret=False), '')
test_eq(store.values(['PULLUP_DEMO_REGISTRY'], secret=False), {})
test_fail(lambda: store.set('  ', 'x'), contains='key is required')
test_fail(lambda: store.set('PULLUP_DEMO_TAG', ''), contains='needs a value')

In [ ]:
#| export
def env_value(store, key, default='', secret=False):
    "One value out of an environment store, or `default` when it is unset or unreadable."
    try: return store.get(key, secret=secret) or default
    except Exception: return default

`env_value` is the reading half of the store for callers that must not stop. Every exception becomes
`default`, including the `EnvError` from a store with no dockeasy behind it. A deploy that cannot
read a region falls back to the one it ships with rather than failing the step that wanted it.

`secret` defaults to False here and True on `EnvStore.get`. The callers of `env_value` read
variables.

In [ ]:
env_value(store, 'PULLUP_DEMO_REGISTRY', 'ghcr.io/fallback')

'ghcr.io/fallback'

In [ ]:
#| hide
class Broken:
    def get(self, key, secret=False): raise EnvError('no store here')
test_eq(env_value(Broken(), 'PULLUP_DEMO_REGISTRY', 'ghcr.io/fallback'), 'ghcr.io/fallback')
test_eq(env_value(Broken(), 'PULLUP_DEMO_REGISTRY'), '')

In [ ]:
#| hide
os.environ.pop('PULLUP_DEMO_REGISTRY', None)
tmp.cleanup()